In [1]:
import langchain
import langchain_ollama # pip install langchain-ollama
print("LangChain version:", langchain.__version__)

LangChain version: 1.2.17


In [2]:
from langchain_ollama import OllamaLLM

## Ollama

Ollama: free, local model.

1. Download model from https://ollama.com/ (macOS: `brew install ollama`)
2. Start the Ollama service: `ollama serve`
3. Pull a model: `ollama pull llama3.1:8b`

### Connect to Ollama

In [3]:
llm = OllamaLLM(model='llama3.1:8b')

In [4]:
response = llm.invoke('What is RAG in AI in 1 sentence?')
print(response)

RAG (Reactor, Augmenter, Generator) is a state-of-the-art AI model architecture that has achieved exceptional results in tasks such as text-to-text and text summarization by using a combination of three components: Reactor, Augmenter, and Generator.


### Create a knowledge base

This simulates documents sources such as PDFs, web pages, or text.

In [5]:
from langchain_core.documents import Document

In [6]:
docs = [
    Document(page_content="LangChain is a framework for building applications powered by large language models."),
    Document(page_content="RAG stands for Retrieval Augmented Generation. It grounds LLM responses in real documents."),
    Document(page_content="Ollama lets you run large language models locally on your own machine for free."),
    Document(page_content="Vector stores store embeddings — numerical representations of text — for similarity search."),
    Document(page_content="LangChain supports many LLM providers including OpenAI, Anthropic, and Ollama."),
]

print(f"{len(docs)} documents loaded")

5 documents loaded


### Convert documents into vectors (embeddings)

In [7]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

In [8]:
embeddings = OllamaEmbeddings(model="llama3.1:8b")
vectorstore = FAISS.from_documents(docs, embeddings)

FAISS stands for Facebook AI Similarity Search, a library developed by Meta for efficiently searching through large collections of vectors (embeddings). In production, FAISS is swapped for hosted vector databases like Pinecone, Weaviate, or ChromaDB.

### Build the retriever

In [9]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

query = "What is RAG?"
retrieved_docs = retriever.invoke(query)

print("Retrieved context:")
for doc in retrieved_docs:
    print("-", doc.page_content)

Retrieved context:
- RAG stands for Retrieval Augmented Generation. It grounds LLM responses in real documents.
- LangChain is a framework for building applications powered by large language models.


### Building the RAG chain

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [11]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [12]:
prompt = ChatPromptTemplate.from_template("""
Answer the question based ONLY on the following context. 
If the answer is not in the context, say "I don't know".

Context: {context}
Question: {question}
""")

In [13]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [14]:
questions = [
    "What is RAG?",
    "What is LangChain?",
    "Who created FAISS?",  # not in our docs — should say I don't know
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}")
    print("---")

Q: What is RAG?
A: RAG (Retrieval Augmented Generation) stands for grounding LLM responses in real documents.
---
Q: What is LangChain?
A: LangChain is a framework for building applications powered by large language models.
---
Q: Who created FAISS?
A: I don't know
---


## File as knowledge base

In [15]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [16]:
loader = TextLoader("../data/ai_glossary.txt")
raw_docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
split_docs = splitter.split_documents(raw_docs)

print(f"Loaded {len(raw_docs)} document, split into {len(split_docs)} chunks")
for i, chunk in enumerate(split_docs):
    print(f"\nChunk {i+1}: {chunk.page_content}")

Loaded 1 document, split into 8 chunks

Chunk 1: LangChain is an open-source framework designed to simplify the creation of applications using large language models.

Chunk 2: RAG stands for Retrieval Augmented Generation. It is a technique that grounds LLM responses in external documents to reduce hallucination.

Chunk 3: Ollama is a tool that allows developers to run large language models locally on their own hardware without needing cloud APIs.

Chunk 4: A vector store is a database that stores embeddings — numerical representations of text — and enables fast similarity search.

Chunk 5: FAISS is a library developed by Meta for efficient similarity search and clustering of dense vectors.

Chunk 6: An embedding is a numerical representation of text in high-dimensional space where semantically similar texts are close together.

Chunk 7: A prompt template is a reusable structure that formats inputs before sending them to a language model.

Chunk 8: Hallucination in AI refers to when a 

In [17]:
vectorstore = FAISS.from_documents(split_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [18]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [19]:
questions = [
    "What is hallucination in AI?",
    "What is FAISS?",
    "What is an embedding?",
    "Who is the CEO of OpenAI?",  # not in docs — should say I don't know
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}")
    print("---")

Q: What is hallucination in AI?
A: Hallucination in AI refers to when a language model generates confident but factually incorrect information.
---
Q: What is FAISS?
A: FAISS is a library developed by Meta for efficient similarity search and clustering of dense vectors.
---
Q: What is an embedding?
A: A numerical representation of text in high-dimensional space where semantically similar texts are close together.
---
Q: Who is the CEO of OpenAI?
A: I don't know.
---


### Save and reload the vector store

To avoid rebuilding vector stire everytime.

In [20]:
vectorstore.save_local("../data/faiss_index")

In [21]:
# reloading
vectorstore_loaded = FAISS.load_local(
    "../data/faiss_index", 
    embeddings,
    allow_dangerous_deserialization=True
)

In [22]:
retriever = vectorstore_loaded.as_retriever(search_kwargs={"k": 2})

In [23]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [24]:
questions = [
    "What is hallucination in AI?",
    "What is FAISS?",
    "What is an embedding?",
    "Who is the CEO of OpenAI?",  # not in docs — should say I don't know
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}")
    print("---")

Q: What is hallucination in AI?
A: Hallucination in AI refers to when a language model generates confident but factually incorrect information.
---
Q: What is FAISS?
A: FAISS is a library.
---
Q: What is an embedding?
A: A numerical representation of text in high-dimensional space where semantically similar texts are close together.
---
Q: Who is the CEO of OpenAI?
A: I don't know.
---
